In [93]:
import io
import re
import requests
import pdfplumber
import pytesseract
import psycopg2 as pg
from pdf2image import convert_from_bytes
import os
from dotenv import load_dotenv

In [2]:
# Save the link to access the PDF
url = "https://s3-fs1.fsn1.your-objectstorage.com/s3-fs1/sign_documents/205049000036123333.pdf"

In [3]:
# Return the response of te link, 
# 200: meaning it can be access
# 404: failed to access the PDF
response = requests.get(url)

In [4]:
# Virtual PDF file - exists only in RAM, not on disk.
# Faster access from RAM memory than disk memory.
# When the program is ended, the memory is released.
pdf_bytes = io.BytesIO(response.content)

In [5]:
# Open the virtual PDF saved in RAM memory as pdf,
# with pdfplumber: closes the virtual file when leaving the notebook block 
with pdfplumber.open(pdf_bytes) as pdf:
    # enumerate(): add a numeric counter in each iteration,
    # where page is an object from pdfplumber representing the loaded page,
    # ready to be manipulated
    for i, page in enumerate(pdf.pages[:2]):
        # Extract the text from the pdfplumber object (reprenting the loaded page)
        text = page.extract_text()
        # Print the numeric counter, which represent each page
        print(i)
        # Print the text from the page
        print(text)

0
Zoho Sign Document ID: 2BF51FA9-CWCSOXJUEQUKJLP6-UKDRBV0_4R1LUGAIKYFJLY4HMI
RMA FORM
To:
P:
E: Warranty Serial #: Date Order #
A:
Account
Item Purchased VIN For Vehicle
Manager
Problem(s) experienced with the ORIGINAL MODULE
ORIGINAL MODULE's Symptoms (e.g: vehicle misfiring, stalling, shutting off):
Was the vehicle starting with the ORIGINAL MODULE?
IF NOT, was the engine receiving any of the following:
YES NO
Fuel Spark Starter Activating
Error codes with the ORIGINAL MODULE (error code number, e.g: P1000, P0300):
Steps taken to diagnose the problem with the ORIGINAL MODULE:
Result from those Steps:
Page 1 of 3
accounts@fs1inc.com 19 Wilbur Street,
(516) 766-2223
www.fs1inc.com Lynbrook, NY, 11563
1
Zoho Sign Document ID: 2BF51FA9-CWCSOXJUEQUKJLP6-UKDRBV0_4R1LUGAIKYFJLY4HMI
RMA FORM
To:
P:
E: Warranty Serial #: Date Order #
A:
Account
Item Purchased VIN For Vehicle
Manager
Problem(s) experienced with the unit from FLAGSHIP ONE
FLAGSHIP ONE MODULE's Symptoms (e.g: vehicle misfiring,

## Page 1

In [69]:
# Return the internal cursor to the beginning of the file.
# When reading the file above, the internal cursor is moved until the end,
# seek() return it to the beginning of the file, so that the next reading starts from the beginning of the file
pdf_bytes.seek(0)

0

In [70]:
# Convert from BytesIO object to PIL.Image of page 1
# dpi (Dots Per Inch): pixels density per inch when converting from image to PDF.
# As higher as dpi is, better is the image quality
images_page1 = convert_from_bytes(pdf_bytes.read(), first_page=1, last_page=1, dpi=600)

In [71]:
# Convert from PIL.Image object to text/string
ocr_text_page1 = pytesseract.image_to_string(images_page1[0])

In [72]:
ocr_text_page1

"Zoho Sign Document ID: 2BF51FA9-CWCSOX] UEQUK] LP6-UKDRBV0O_4R1LUGAIKY FJ LY 4HMI\n\nXN FLAGSHIP\n\nTo: H10041339 Elliot Langlinais\n\nRMA FORM\n\nP: 3373036660\n\nE:  Elliotlanglinais@yahoo.com\n2026-04-15 Web\n\nA: 2831S Darla Ave., Gonzales, 70737, United States 1108803\nAccount .\nItem Purchased | VIN For Vehicle\nManager\nJayson C 05150941AB | 2015 Jeep Patriot 2.4L ECM Engine Computer PCM ECU 1C4NJPBB8FD431200\n\nProgrammed Plug&Play | 05150941AB\n\nORIGINAL MODULE's Symptoms (e.g: vehicle misfiring, stalling, shutting off):\n\nRough idle, Battery light would trigger and the car would go into limp mode with zero peddle response. After restarting\nthe car all codes would clear for a random amount of time before proceeding with the same Symptoms. Code given\nU0401. Permanent\n\nWas the vehicle starting with the ORIGINAL MODULE?\n\nIF NOT, was the engine receiving any of the following:\n\n[ ] NO [] Fuel [] Spark [] Starter A ctivating\n\nError codes with the ORIGINAL MODULE (error 

In [73]:
# Seach for the automake name in the page 1
automake_match = re.search(r'VIN For Vehicle[^\n]*\n.*?\d{4}(?!-)\s+([A-Za-z]+)', ocr_text_page1, re.DOTALL)

In [74]:
# Select the 2nd group on the string where the automaker name is located 
make_name = automake_match.group(1)

In [10]:
# re.search: search for the pattern in the text
# :\n\n: 
# (.*?): capture all text after \n\n until the next \n\n
# (?=\n\n|\Z): regex to limit where the search stops, next blank line
orig_dtc_match = re.search(r'Error codes with the ORIGINAL MODULE[^\n]*:\n\n(.*?)(?=\n\n|\Z)', ocr_text_page1, re.DOTALL)

In [11]:
orig_dtc_match.group(1)

'U0401. Permanent'

In [12]:
# Condition to get the 2nd part of the string if text was captured
if orig_dtc_match:
    # group(1): returns the 2nd part of the string, which is end of the string
    # and remove the spaces 
    orig_dtcs = orig_dtc_match.group(1).strip()
else:
    "not captured"

In [13]:
orig_dtc = re.findall(r'[PCBU][0-9A-F]{4}', orig_dtcs, re.IGNORECASE)

In [14]:
for dtc in orig_dtc:
    print(f"{dtc}")
    print("\n")

U0401




## Page 2

In [52]:
# Return the internal cursor to the beginning of the file.
# When reading the file above, the internal cursor is moved until the end,
# seek() return it to the beginning of the file, so that the next reading starts from the beginning of the file
pdf_bytes.seek(0)

0

In [53]:
# Convert from BytesIO object to PIL.Image
images_page2 = convert_from_bytes(pdf_bytes.read(), first_page=2, last_page=2, dpi=600)

In [54]:
# Convert from PIL.Image to string/text
ocr_text_page2 = pytesseract.image_to_string(images_page2[0])

In [55]:
ocr_text_page2

"Zoho Sign Document ID: 2BF51FA9-CWCSOX] UEQUK] LP6-UKDRBV0O_4R1LUGAIKY FJ LY 4HMI\n\nXN FLAGSHIP\n\nTo: H10041339 Elliot Langlinais\n\nRMA FORM\n\nP: 3373036660\n\nE: Elliotlanglinais@yahoo.com\n2026-04-15 Web\n\nA: 2831 S Darla Ave., Gonzales, 70737, United States 1108803\nAccount .\nItem Purchased | VIN For Vehicle\nManager\nJayson C 05150941AB | 2015 Jeep Patriot 2.4L ECM Engine Computer PCM ECU Programmed Plug&Play | 1C4NJPBB8FD431200\n\n05150941AB\n\nFLAGSHIP ONE MODULE's Symptoms (e.g: vehicle misfiring, stalling, shutting off):\n\nDisconnected and removed battery, Installed new module, Replaced battery. Started the vehicle. per start up had a rough idle, miss firing, battery light was on and check engine\nlight was flashing. Turned off vehicle. Erased codes off ecm. restarted the vehicle. Same issues.\n\nDoes the vehicle start with the FLAGSHIP ONE MODULE?\n\nIF NOT, was the engine receiving any of the following:\n\nL | NO [] Fuel [] Spark [] Starter Activating\n\nError codes w

In [56]:
# re.search: search for the pattern in the text
# (.*?): capture all text after \n\n until the next \n\n
# (?=\n\n|\Z): regex to limit where the search stops, next blank line
fs1_dtc_match = re.search(r'Error codes with FLAGSHIP ONE MODULE[^\n]*:\n\n(.*?)(?=\n\n|\Z)', ocr_text_page2, re.DOTALL)

In [57]:
if fs1_dtc_match:
    fs1_dtcs = fs1_dtc_match.group(1).strip()
else:
    "not captured"

In [59]:
fs1_dtc = re.findall(r'[PCBU][0-9A-F]{4}', fs1_dtcs, re.IGNORECASE)

In [61]:
type(fs1_dtc)

list

In [27]:
for dtc in fs1_dtc:
    print(f"{dtc}")

U0401
P0301
P0300
P219A
P0123
P2122
P2122
P2127
P0108


## Show dtc description

In [94]:
# Create the connection with the database from the env vars
conn = pg.connect(
    host=os.getenv("HOST_NAME"),
    port=os.getenv("PORT_NUMBER"),
    dbname=os.getenv("DB_NAME"),
    user=os.getenv("USER_NAME"),
    password=os.getenv("PASSWORD")
)

AVISO:  o banco de dados "prescreen_diag_data_api" possui uma não correspondência de versão de ordenação
DETAIL:  The database was created using collation version 1539.5,1539.5, but the operating system provides version 1541.2,1541.2.
HINT:  Rebuild all objects in this database that use the default collation and run ALTER DATABASE prescreen_diag_data_api REFRESH COLLATION VERSION, or build PostgreSQL with the right library version.


In [95]:
load_dotenv("/mnt/e/language_projects/language_projects/Python/Flagship_1/dtc-form-automation/.env", override=True)

True

In [96]:
# psycopg2 object responsible for execute queries
cur = conn.cursor()

In [64]:
# Create a list with the file names containing the dtcs
automaker_db_tables_names_dict = {
    "acura_dtcs": "Acura",
    "audi_dtcs": "Audi",
    "bmw_dtcs": "BMW",
    "buick_dtcs": "Buick",
    "cadillac_dtcs": "Cadillac",
    "chevrolet_dtcs": "Chevrolet",
    "chrysler_dtcs": "Chrysler",
    "dodge_dtcs": "Dodge",
    "ford_dtcs": "Ford",
    "generic_dtcs": "Generic",
    "geo_dtcs": "Geo",
    "gmc_dtcs": "GMC",
    "honda_dtcs": "Honda",
    "hyundai_dtcs": "Hyundai",
    "hummer_dtcs": "Hummer",
    "infiniti_dtcs": "Infiniti",
    "isuzu_dtcs": "Isuzu",
    "jaguar_dtcs": "Jaguar",
    "jeep_dtcs": "Jeep",
    "kia_dtcs": "Kia",
    "land_rover_dtcs": "Land Rover",
    "lexus_dtcs": "Lexus",
    "mazda_dtcs": "Mazda",
    "mercedes_benz_dtcs": "Mercedes-Benz",
    "mini_dtcs": "Mini",
    "mitsubishi_dtcs": "Mitsubishi",
    "nissan_dtcs": "Nissan",
    "oldsmobile_dtcs": "Oldsmobile",
    "pontiac_dtcs": "Pontiac",
    "saturn_dtcs": "Saturn",
    "subaru_dtcs": "Subaru",
    "toyota_dtcs": "Toyota",
    "volkswagen_dtcs": "Volkswagen"
}

In [88]:
automaker_table = None

# Loop to iterate over the dict with the table names and automakers
for table_name, maker_name in automaker_db_tables_names_dict.items():
    # Condition using case insensitive to confirm when the automaker matches
    # with the name in the dictionary
    if maker_name.lower() == make_name.lower():
        # Assign the table name to a variable 
        automaker_table = table_name
        break

In [97]:
dtc_descriptions_list = []

for dtc_code in fs1_dtc:
    description = None

    # Search in the automaker's table
    if automaker_table:
        cur.execute(
            f"SELECT description FROM {automaker_table} WHERE LOWER(code) = LOWER(%s)",
            (dtc_code,)
        )
        result = cur.fetchone()
        if result:
            description = result[0]

    # If not found, search in generic_dtcs
    if not description:
        cur.execute(
            "SELECT description FROM generic_dtcs WHERE LOWER(code) = LOWER(%s)",
            (dtc_code,)
        )
        result = cur.fetchone()
        if result:
            description = result[0]
    
    # NOT FOUND MESSAGE if not found in any table
    if not result:
        description = f"{dtc_code} DTC NOT IN THE DATABASE"

    dtc_descriptions_list.append({"code": dtc_code, "description": description}) 

In [98]:
dtc_descriptions_list

[{'code': 'U0401', 'description': 'U0401 DTC NOT IN THE DATABASE'},
 {'code': 'P0301', 'description': 'Cylinder 1 Misfire Detected'},
 {'code': 'P0300', 'description': 'Random/Multiple Cylinder Misfire Detected'},
 {'code': 'P219A', 'description': 'Bank 1 Air/Fuel Ratio Imbalance'},
 {'code': 'P0123',
  'description': 'Throttle Position Sensor/Switch A Circuit High Input'},
 {'code': 'P2122',
  'description': 'Throttle/Pedal Position Sensor/Switch "D" Circuit Low'},
 {'code': 'P2122',
  'description': 'Throttle/Pedal Position Sensor/Switch "D" Circuit Low'},
 {'code': 'P2127',
  'description': 'Throttle/Pedal Position Sensor/Switch "E" Circuit Low'},
 {'code': 'P0108',
  'description': 'Manifold Absolute Pressure/Barometric Pressure Circuit High Input'}]